<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefoerdert vom Bundesministerium fuer Forschung, Technologie und Raumfahrt.">
</div>
<h1> Workshop: Rapid Prototyping via Vibe Coding
<h2> Part 1 &mdash; Configure your Environment
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
    <h2> Part 1 &mdash; Configure your Environment</h2>
</div>

The previous notebook covered how to *use* Claude Code well. This one covers how to *configure* it so those good habits are the default &mdash; for you, your team, and anyone who clones the repo.

Eight mechanisms, ordered from highest-frequency to lowest:

1. `CLAUDE.md` &mdash; persistent per-project context.
2. **Permissions** &mdash; what Claude may do without asking.
3. **CLI tools** &mdash; the most context-efficient way to reach external services.
4. **MCP servers** &mdash; structured integrations with APIs and data sources.
5. **Hooks** &mdash; deterministic shell commands wired into Claude's lifecycle.
6. **Skills** &mdash; domain knowledge and reusable workflows.
7. **Subagents** &mdash; delegated workers with their own context window.
8. **Plugins** &mdash; bundled skills/hooks/MCP from Anthropic or the community.

The companion files referenced below live in [`assets/`](assets/) next to this notebook.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. CLAUDE.md</h3>
</div>

`CLAUDE.md` is a special file Claude reads at the **start of every conversation**. It's persistent context Claude cannot infer from code alone &mdash; Bash commands, code-style rules, workflow etiquette, and quirks.

#### Generate a starter

```bash
/init
```

The `/init` slash command analyses your codebase and writes a first `CLAUDE.md` based on what it finds (build system, test framework, code patterns). Treat the output as a starting point, not gospel.

#### What to include vs. exclude

| Include | Exclude |
|---------|---------|
| Bash commands Claude can't guess | Anything Claude can figure out by reading code |
| Code-style rules that differ from defaults | Standard language conventions Claude already knows |
| Testing instructions and preferred test runners | Detailed API documentation (link instead) |
| Repository etiquette (branch naming, PR conventions) | Information that changes frequently |
| Architectural decisions specific to your project | Long explanations or tutorials |
| Developer-environment quirks (required env vars) | File-by-file descriptions of the codebase |
| Common gotchas or non-obvious behaviours | Self-evident practices like "write clean code" |

> **Keep it short.** For every line, ask: *"Would removing this cause Claude to make mistakes?"* If not, cut it.
>
> Bloated `CLAUDE.md` files cause Claude to ignore your actual instructions because the real rules get lost in the noise.

#### Where `CLAUDE.md` files live

| Location | Applies to | Checked into git? |
|----------|-----------|-------------------|
| `~/.claude/CLAUDE.md` | every Claude session on your machine | no (personal) |
| `./CLAUDE.md` (project root) | this project, everyone who clones it | **yes** |
| `./CLAUDE.local.md` | this project, you only | no (add to `.gitignore`) |
| `./subdir/CLAUDE.md` | loaded on demand when Claude works in `subdir/` | yes |
| `../../CLAUDE.md` (parent) | monorepos &mdash; Claude pulls both root and leaf | yes |

#### Imports

`CLAUDE.md` can pull in other files:

```markdown
See @README.md for project overview and @package.json for available npm commands.

# Additional Instructions
- Git workflow: @docs/git-instructions.md
- Personal overrides: @~/.claude/my-project-instructions.md
```

#### Example

See [`assets/CLAUDE.example.md`](assets/CLAUDE.example.md) for a concise, annotated template you can copy into a real project.

#### Tuning

- Use emphasis (`IMPORTANT`, `YOU MUST`) sparingly &mdash; it lifts a rule above the noise but loses effect if overused.
- If Claude keeps doing something your `CLAUDE.md` says not to do, the file is probably too long and the rule is getting lost. Prune before you add more.
- If Claude *asks* you things that are already in `CLAUDE.md`, the phrasing may be ambiguous. Rewrite.
- Treat it like code: review when things go wrong, prune regularly, **check it into git** so the team compounds the value.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Permissions</h3>
</div>

By default Claude Code asks for permission before it writes files, runs Bash commands, or uses MCP tools. Safe, but after the tenth approval you're not reviewing any more &mdash; you're clicking through. Three ways to reduce the friction:

#### 1. Auto mode

A separate classifier model reviews each command and only blocks risky ones (scope escalation, unknown infra, hostile-content-driven actions). Toggle with **Shift+Tab** until the banner shows `auto mode`, or start a session with:

```bash
claude --permission-mode auto
```

Best when you trust the direction of a task but don't want to click through every step.

#### 2. Permission allowlists

Add specific tools you know are safe to `.claude/settings.json`:

```json
{
  "permissions": {
    "allow": [
      "Bash(npm run lint)",
      "Bash(npm test:*)",
      "Bash(git status)",
      "Read",
      "Write(./src/**)"
    ]
  }
}
```

Run `/permissions` inside Claude Code to see what's currently allowed and to add rules interactively.

See [`assets/settings.example.json`](assets/settings.example.json) for a fuller template with a hook alongside.

#### 3. Sandboxing

OS-level isolation that restricts filesystem and network access, letting Claude work more freely inside the box.

```bash
/sandbox
```

Most useful when Claude is running long experiments or destructive-seeming refactors you want caged.

#### Which to use when

| You want... | Use |
|-------------|-----|
| Routine work without constant prompts | auto mode |
| A small set of commands to never prompt | `permissions.allow` |
| Hard guarantees that Claude can't break things | `/sandbox` |

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. CLI tools</h3>
</div>

CLI tools are the most **context-efficient** way to reach external services &mdash; Claude calls `gh`, `aws`, `gcloud`, `sentry-cli`, `kubectl`, and learns their interface from `--help` instead of needing a custom integration.

#### Install the ones you use

```bash
# GitHub
brew install gh   # or: https://github.com/cli/cli

# AWS
brew install awscli

# GCP
brew install --cask google-cloud-sdk

# Sentry (errors)
brew install getsentry/tools/sentry-cli
```

If you use GitHub at all, `gh` is the highest-leverage install: Claude can open PRs, comment on issues, fetch check statuses, and read diffs without hitting rate limits.

#### Teach Claude a new CLI

Claude is quick to learn tools it doesn't already know. A reliable prompt pattern:

```text
Use `foo-cli --help` to learn about the foo tool.
Then use it to solve A, B, and C.
Summarise what you learned in two sentences.
```

This pattern pays off best when the tool's help is self-contained (one screen of output).

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. MCP servers</h3>
</div>

Where CLI tools are for *unix-shaped* services, **MCP servers** (Model Context Protocol) are for structured integrations &mdash; they expose typed tools Claude can call directly.

```bash
claude mcp add <name> -- <command to run the server>
```

#### Common things to wire up

| Server | What it unlocks |
|--------|-----------------|
| Linear / Jira / Asana | Read & create issues, attach PRs, triage backlogs |
| Notion / Confluence | Read internal docs and specs |
| Postgres / BigQuery / DuckDB | Safe read-only queries against your data warehouse |
| Figma | Reference designs directly in prompts |
| Grafana / Sentry / Datadog | Look up alerts, traces, error rates |
| GitHub (as MCP) | Richer than `gh` for PR-review-heavy workflows |

Run `/mcp` inside Claude Code to see what's wired up. `claude mcp list` shows the same thing from the shell.

#### Tip

For each new MCP server, check whether a CLI already does the job cheaply. MCP is richer but costs more context; `gh` + a well-written prompt often beats a GitHub MCP server for everyday tasks.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Hooks</h3>
</div>

**Hooks** run shell commands automatically at specific points in Claude's workflow. Unlike `CLAUDE.md` rules which are *advisory*, hooks are *deterministic* &mdash; the shell always runs them.

#### When to reach for a hook

| Use a hook when... | Use `CLAUDE.md` when... |
|--------------------|------------------------|
| The action must happen every single time with zero exceptions | Advisory: "prefer this style" |
| You can express it as a shell command | It needs Claude's judgement |
| It's cheap (milliseconds) | It's context, not automation |

Classic hooks:

- Run `eslint` / `ruff` / `mypy` after every edit.
- Block writes to `migrations/` without an explicit flag.
- Auto-stage a changelog entry when a file under `src/features/` is modified.
- Post a Slack notification when Claude finishes a long task.

#### Where they live

`.claude/settings.json`, under the `hooks` key. Run `/hooks` to see what's currently wired up.

#### Let Claude write the hook

Hooks are fiddly enough that Claude writing them for you is almost always the right move:

```text
write a PostToolUse hook that runs `npm run lint -- --fix` after every Edit.
only run it for files under src/. add it to .claude/settings.json.
```

See [`assets/settings.example.json`](assets/settings.example.json) for a minimal example.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. Skills</h3>
</div>

**Skills** extend Claude with information or workflows specific to your project, team, or domain. Claude applies them *automatically* when relevant (based on the skill's description), or you invoke them directly with `/skill-name`.

#### Where they live

```
.claude/skills/<skill-name>/SKILL.md
```

A skill is just a markdown file with frontmatter. The simplest flavour: domain knowledge.

```markdown
---
name: api-conventions
description: REST API design conventions for our services
---
# API Conventions
- Use kebab-case for URL paths.
- Use camelCase for JSON properties.
- Always include pagination for list endpoints.
- Version APIs in the URL path (/v1/, /v2/).
```

See [`assets/skills/api-conventions/SKILL.md`](assets/skills/api-conventions/SKILL.md) for this exact example as a file.

#### Workflow skills

Skills can also define repeatable multi-step workflows invoked by name:

```markdown
---
name: fix-issue
description: Fix a GitHub issue
disable-model-invocation: true
---
Analyze and fix the GitHub issue: $ARGUMENTS.

1. Use `gh issue view` to get the issue details.
2. Understand the problem described.
3. Search the codebase for relevant files.
4. Implement the fix.
5. Write tests to verify.
6. Ensure code passes linting and typechecking.
7. Create a descriptive commit message.
8. Push and open a PR.
```

Invoke with `/fix-issue 1234`. Use `disable-model-invocation: true` for workflows that have side effects and should only run when you explicitly ask &mdash; prevents Claude from "helpfully" invoking them on its own.

#### Skills vs. `CLAUDE.md`

> Put **broadly applicable** context in `CLAUDE.md` (it loads every session).
> Put **sometimes-relevant** knowledge in a skill (it loads on demand).

This is the single most important optimisation for keeping the context window lean.

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>7. Subagents</h3>
</div>

**Subagents** are delegated workers. They run in their own context window with their own allowed tool-set, and report back a summary to your main conversation.

Two big wins:
1. Exploration doesn't pollute your main context &mdash; Claude reads 40 files, summarises, you get the summary.
2. You can enforce a different skill-set (a security-reviewer subagent with only `Read, Grep, Glob`, no `Edit`).

#### Where they live

```
.claude/agents/<agent-name>.md
```

Example (see [`assets/agents/security-reviewer.md`](assets/agents/security-reviewer.md)):

```markdown
---
name: security-reviewer
description: Reviews code for security vulnerabilities
tools: Read, Grep, Glob, Bash
model: opus
---
You are a senior security engineer. Review code for:
- Injection vulnerabilities (SQL, XSS, command injection)
- Authentication and authorization flaws
- Secrets or credentials in code
- Insecure data handling

Provide specific line references and suggested fixes.
```

#### Invoking one

Tell Claude explicitly:

```text
use a subagent to review this code for security issues
```

Claude picks the best-matching agent (by description) and delegates.

#### When to use a subagent vs. a skill

| Task shape | Use |
|-----------|-----|
| *"Go look at X files and tell me about Y"* (read-heavy, may bloat main context) | Subagent |
| *"How should I format an API response?"* (knowledge lookup) | Skill |
| *"Do these eight steps to ship a fix"* (workflow) | Skill |
| *"Review this PR with fresh eyes"* (need a clean context for judgement) | Subagent |

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>8. Plugins</h3>
</div>

**Plugins** are pre-packaged bundles of skills, hooks, subagents, and MCP servers. One install gives you a coherent set of capabilities.

```bash
/plugin                   # browse the marketplace interactively
/plugin install <name>    # install a specific plugin
claude plugin list        # see what's installed
```

The rest of this workshop's Part 1 is, essentially, a tour of plugins and plugin-like tools:

- **`03_*`** &mdash; the [`docling-toolkit`](https://github.com/docling-project/docling) plugin (document extraction).
- **`05_*`** &mdash; the [`context-mode`](https://github.com/mksglu/context-mode) plugin (tool-output sandboxing).
- **`06_*`** &mdash; the [`caveman`](https://github.com/JuliusBrussee/caveman) plugin (output-token compression).

#### Plugins worth a look outside this workshop

- **Code-intelligence plugins** &mdash; if you work in a typed language (TS, Rust, Go, Java, Swift), install one. They give Claude precise symbol navigation (*"find all callers of `Foo.bar`"*) and automatic error detection after edits. Enormous quality-of-life uplift.
- **Framework plugins** (React, Django, Rails, etc.) &mdash; encode idioms so Claude generates code that matches the framework's conventions.

#### Picking between skills / subagents / hooks / MCP / plugins

Rule of thumb: **skills for knowledge, subagents for isolation, hooks for determinism, MCP for structured integrations, plugins for bundled capabilities.**

<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>9. Wrap-up</h3>
</div>

You've now seen the full configuration surface of Claude Code. In practice you won't touch all of it on day one &mdash; pick:

1. Write a `CLAUDE.md` (**always**).
2. Add 2&ndash;3 permissions entries for your most-repeated commands.
3. Install `gh` if you haven't.

Everything else (hooks, skills, subagents, MCP, plugins) is worth adopting the moment you notice you're repeating yourself.

On to the plugins: [`03_docling_toolkit.ipynb`](03_docling_toolkit.ipynb).